# Interpolation
This notebook runs a linear interpolation of SLA data in the month of Jan 2020.

In [1]:
#designate GPU for work
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import pandas as pd
import numpy as np
import time

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/home/mhen/.local/share/fonts/IBMPlexSerif-Regular.ttf"
font_prop = fm.FontProperties(fname=font_path)

plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["font.sans-serif"] = [font_prop.get_name()]

from GPSat.utils import WGS84toEASE2
from GPSat.local_experts import LocalExpertOI, get_results_from_h5file
from GPSat.postprocessing import smooth_hyperparameters, glue_local_predictions_1d

2026-05-25 09:36:56.077226: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 09:36:57.059067: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-25 09:36:57.059143: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-25 09:36:57.245470: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-25 09:36:57.602232: I tensorflow/core/platform/cpu_feature_guar

In [3]:
#load preprocessed data
#replace path
data_df = pd.read_csv('/home/mhen/geol0069_data/jan20.csv')
#keep data from january
data_df = data_df.loc[data_df['date_string'].str.contains('2020-01', na=False)]

With the data preprocessed and loaded, we can now begin the experiment. We create two interpolation functions, both linear and GPSat, with held-out data at a ratio of 80/20 train-to-test. The test data will be used in the results notebook. For now, we look at the interpolation of SLA along the track given the training lead data.

### Linear

The linear interpolation functions according to two steps. First, all SLA values are smoothed to the mean of values within 100km. This gives smoothed data $z$. Next, the $z$ value at all points is determined by the following equation:

$$ z = z_0 + (z_1 - z_0) \frac{d - d_0}{d_1-d_0} $$

Where $z_0$ and $z_1$ are the SLA at the nearest training leads either side of the point, $d_0$ and $d_1$ are the along-track distances of the leads, and $d$ is the along-track distance of the interpolated point.

### GPSat

GPSat is a more complex method of interpolation involving Gaussian processes. A basic explanation of the way the package interpolates along a track can be found at https://cpomucl.github.io/GPSat/notebooks/gp_regression.html.
The interpolation ouputs a distribution along which the data lie, with a mean and a variance, allowing us to plot the mean and a 95% confidence interval in the results.

In [4]:
def random_holdout_test(data_df, holdout_fraction=0.2, seed=42, gpsat_experts=128, file_prefix='jan20', replace_previous=False):
    '''
    Selects test and train datapoints and feeds them into linear and GPSat along-track interpolations.
    Saves data in csvs.'''
    rng = np.random.default_rng(seed)

    #separate into tracks and iterate over each
    tracks = data_df['track'].unique()
    
    results_df = pd.DataFrame()
    results_metrics_df = pd.DataFrame()
    plot_data_df = pd.DataFrame()
    for i, track in enumerate(tracks):
        print(f'Interpolating track {track}, {i+1} of {len(tracks)}...')
        track_df = data_df.loc[data_df['track']==track].copy()
        #locate leads
        leads_df = track_df.loc[track_df['lead_mask']==1.0].copy()
        #create test (holdout) indices of length 1 or of holdout fraction, whichever is larger
        n_test = max(1, int(len(leads_df) * holdout_fraction))

        #exclude tracks with insufficient leads
        if len(leads_df) < 5:
            print(f"Insufficient leads ({len(leads_df)}) for random holdout, skipping...")
            continue

        #find lead indices and randomly select 80% for training
        leads_idx = leads_df.index
        test_lead_idx = rng.choice(leads_df.index, size=n_test, replace=False)
        train_lead_idx = np.array(leads_df.index.difference(test_lead_idx))
        track_df['train_mask'] = track_df.index.isin(train_lead_idx)


        # Compute distances before interpolation - nearest distance to a train lead of each test lead
        lead_D = leads_df.loc[leads_idx, 'dist_along_track'].to_numpy()
        train_D = leads_df.loc[train_lead_idx, 'dist_along_track'].to_numpy()
        test_D = leads_df.loc[test_lead_idx, 'dist_along_track'].to_numpy()
        if train_D.size > 0 and test_D.size > 0:
            nearest_dists_km = np.min(np.abs(test_D[:, None] - train_D[None, :]), axis=1) / 1000.0
        else:
            nearest_dists_km = np.array([])

        #run linear and gpsat interpolation
        start = time.perf_counter()
        result_track_lin = linear_interpolation(track_df, train_lead_idx)
        end = time.perf_counter()
        lin_time = end - start
        result_track_gpsat, gpsat_time = gpsat_interpolation(track_df, train_lead_idx, max_experts = gpsat_experts, replace_previous=replace_previous)

        required_cols = ['f_gpsat_SMOOTHED', 'f_var_gpsat_SMOOTHED']
        
        if (
            result_track_gpsat is None
            or result_track_gpsat.empty
            or not all(col in result_track_gpsat.columns for col in required_cols)
        ):
            print("GPSat output invalid or incomplete, skipping...")
            continue

        result_track = result_track_lin.join(result_track_gpsat[['f_gpsat_SMOOTHED','f_var_gpsat_SMOOTHED']])

        #find test values - withheld leads
        linear_preds = result_track.loc[test_lead_idx,'f_lin'].to_numpy()
        gpsat_preds = result_track.loc[test_lead_idx, 'f_gpsat_SMOOTHED'].to_numpy()
        gpsat_var = result_track.loc[test_lead_idx,'f_var_gpsat_SMOOTHED'].to_numpy()
        targets = result_track.loc[test_lead_idx, 'z'].to_numpy()
        dist_along_track = result_track.loc[test_lead_idx, 'dist_along_track'].to_numpy()

        print(f"Test NaNs — Linear: {np.sum(~np.isfinite(linear_preds))}, GPSat: {np.sum(~np.isfinite(gpsat_preds))}")
        both_valid = (np.isfinite(linear_preds) & np.isfinite(gpsat_preds) & np.isfinite(targets))
        if np.sum(both_valid) == 0:
            print(f"FAILED: No valid predictions from both methods on random holdout")
            continue

        #find metrics for the prediction
        lin_res = linear_preds[both_valid] - targets[both_valid]
        gp_res = gpsat_preds[both_valid] - targets[both_valid]

        def _r2(a, b):
            return np.corrcoef(a, b)[0,1]**2 if len(a) > 1 else np.nan
        
        # Calculate GPSat uncertainty at test leads for calibration check
        if gpsat_var is not None:
            gpsat_std = np.sqrt(gpsat_var[both_valid])
            avg_gpsat_uncertainty = float(np.mean(gpsat_std))
        else:
            avg_gpsat_uncertainty = None
        
        # Distance and lead spacing stats
        dist_mean = float(np.nanmean(nearest_dists_km)) if nearest_dists_km.size else np.nan
        dist_median = float(np.nanmedian(nearest_dists_km)) if nearest_dists_km.size else np.nan
        dist_max = float(np.nanmax(nearest_dists_km)) if nearest_dists_km.size else np.nan
        dist_min = float(np.nanmin(nearest_dists_km)) if nearest_dists_km.size else np.nan
        spacing_km = np.diff(lead_D) / 1000.0 if lead_D.size > 1 else np.array([])
        sp_med = float(np.nanmedian(spacing_km)) if spacing_km.size else np.nan
        sp_p90 = float(np.nanpercentile(spacing_km, 90)) if spacing_km.size else np.nan
        sp_max = float(np.nanmax(spacing_km)) if spacing_km.size else np.nan

        #add metrics to dataframe
        metrics_dict = {
            'track' : track,
            'n_predictions': int(np.sum(both_valid)),
            'n_train': int(len(train_lead_idx)),
            'n_test': int(len(test_lead_idx)),
            'nearest_train_km': {
                'mean': dist_mean, 'median': dist_median, 'max': dist_max, 'min': dist_min
            },
            'lead_spacing_km': {
                'median': sp_med, 'p90': sp_p90, 'max': sp_max
            },
            'linear': {
                'rmse': float(np.sqrt(np.mean(lin_res**2))),
                'bias': float(np.mean(lin_res)),
                'mae': float(np.mean(np.abs(lin_res))),
                'r2': float(_r2(linear_preds[both_valid], targets[both_valid])),
                'interp_time' : lin_time
            },
            'gpsat_along': {
                'rmse': float(np.sqrt(np.mean(gp_res**2))),
                'bias': float(np.mean(gp_res)),
                'mae': float(np.mean(np.abs(gp_res))),
                'r2': float(_r2(gpsat_preds[both_valid], targets[both_valid])),
                'avg_uncertainty': avg_gpsat_uncertainty,
                'interp_time' : gpsat_time
                },
        }
        plot_data_dict = {
            'track' : track,
            'linear_preds': linear_preds[both_valid].copy(),
            'gpsat_preds': gpsat_preds[both_valid].copy(),
            'gpsat_var' : gpsat_var[both_valid].copy(),
            'targets': targets[both_valid].copy(),
            'nearest_dists_km': nearest_dists_km[both_valid] if nearest_dists_km.size > 0 else np.array([]),
            'linear_residuals': lin_res.copy(),
            'gpsat_residuals': gp_res.copy(),
            'dist_along_track': dist_along_track[both_valid].copy()
        }


        track_plot_data_df = pd.DataFrame(plot_data_dict)
        plot_data_df = pd.concat([plot_data_df, track_plot_data_df])

        results_df = pd.concat([results_df, result_track])
        track_metrics_df = pd.json_normalize([metrics_dict])
        results_metrics_df = pd.concat([results_metrics_df, track_metrics_df])

        print(f'Track interpolated!')
        #save per track to allow for analysing results quickly. Lots of read/write might be unnecessary, in which case
        #move these lives to outside the loop.
            #save files 
        print('Saving...')
        plot_data_df.to_csv(f'/home/mhen/geol0069_data/results/{file_prefix}_lin_gpsat_plot_data.csv')
        results_df.to_csv(f'/home/mhen/geol0069_data/results/{file_prefix}_lin_gpsat_results.csv')
        results_metrics_df.to_csv(f'/home/mhen/geol0069_data/results/{file_prefix}_lin_gpsat_metrics.csv')
    
    print('All tracks interpolated!')

    return plot_data_df, results_df, results_metrics_df


def linear_interpolation(track_df, train_lead_idx, filter_width_km=100):
    '''
    Interpolates along a track linearly.'''
    obs_df = track_df[['z', 'dist_along_track']].copy()
    obs_df = obs_df.dropna(subset=['dist_along_track'])
    train_leads_df = track_df.loc[train_lead_idx]
    train_leads_df = train_leads_df.dropna(subset=['dist_along_track'])

    # Filter leads by height
    height_valid = (train_leads_df['z']>= -0.5) & (train_leads_df['z'] <= 0.5)
    train_leads_df = train_leads_df.loc[height_valid]

    #terminate if too short for linear interpolation
    if len(train_leads_df) < 2:
        print('Too few leads for linear interpolation, returning NaN')
        obs_df['f_lin'] = np.nan
        track_df = pd.merge(track_df, obs_df['f_lin'], left_index=True, right_index=True)
        
        return track_df
    
    # apply 100 km box filter to smooth lead observations
    smoothed_obs_df = obs_df.copy()

    # For each lead point, smooth with nearby leads within filter_width_km
    for idx, lead in train_leads_df.iterrows():
        lead_distances = np.abs(train_leads_df['dist_along_track'] - train_leads_df['dist_along_track'].loc[idx])
        within_window = lead_distances <= (filter_width_km * 1000 / 2)  # ±50 km    
        if np.sum(within_window) > 0:
            smoothed_obs_df.loc[idx] = (train_leads_df['z'][within_window]).mean()

    # linear interpolation between smoothed lead values - covers all obs points
    obs_df['f_lin'] = np.nan

    


    # lin interp function
    for i0, i1 in zip(train_leads_df.index[:-1], train_leads_df.index[1:]):
        v0, v1 = smoothed_obs_df.loc[i0, 'z'], smoothed_obs_df.loc[i1, 'z']
        d0, d1 = train_leads_df.loc[i0, 'dist_along_track'], train_leads_df.loc[i1, 'dist_along_track']
        filt = (obs_df['dist_along_track'] >= d0) & (obs_df['dist_along_track'] <= d1)

        d = obs_df.loc[filt, 'dist_along_track']
        if d1 == d0:
            obs_df.loc[filt, 'f_lin'] = v0
        else:
            obs_df.loc[filt, 'f_lin'] = v0 + (v1-v0)*(d-d0)/(d1-d0)
            
    # flat extrapolation beyond the first/last training lead
    first_idx, last_idx = train_leads_df.index[0], train_leads_df.index[-1]
    first_d, last_d = train_leads_df.loc[first_idx, 'dist_along_track'], train_leads_df.loc[last_idx,'dist_along_track']

    obs_df.loc[obs_df['dist_along_track'] <= first_d, 'f_lin'] = smoothed_obs_df.loc[first_idx, 'z']
    obs_df.loc[obs_df['dist_along_track'] >= last_d, 'f_lin'] = smoothed_obs_df.loc[last_idx, 'z']


    track_df = pd.merge(track_df, obs_df['f_lin'], left_index=True, right_index=True)

    return track_df

def gpsat_interpolation(track_df, train_lead_idx, max_experts=128, replace_previous=False):
    '''
    Interpolates along a track using GPSat.
    '''
    
    obs_df = track_df['z'].copy()
    train_leads_df = track_df.loc[train_lead_idx]

    # Filter leads by height
    height_valid = (train_leads_df['z']>= -0.5) & (train_leads_df['z'] <= 0.5)
    train_leads_df = train_leads_df.loc[height_valid]

    #if too short, terminate and return NaN
    if len(train_leads_df)<3:
        print('Too few leads for GPSat interpolation, returning NaN')
        track_df['f_gpsat'] = np.nan
        
        return track_df, np.nan
    
    #select experts
    experts_df = select_experts(train_leads_df, max_experts = max_experts)

    if len(experts_df) < 2:
        print('Too few experts for GPSat, returning NaN')
        track_df['f_gpsat'] = np.nan
        
        return track_df, np.nan
    
    #run gpsat interpolation
    track = track_df['track'].iloc[0]

    store_path = f'/home/mhen/geol0069_data/gpsat/jan20/{track}.h5'

    #remove previous file if set like that, don't run interp if set to keep file and file exits
    if replace_previous == True:
        run_interpolation=True
        if os.path.exists(store_path):
            print('Track file already exists, REMOVING...')
            os.remove(store_path)
        if not os.path.exists(store_path):
            print('Track file does not already exist. CREATING...')

    if replace_previous == False:
        if os.path.exists(store_path):
            print('Track file already exists, KEEPING...')
            run_interpolation=False
        if not os.path.exists(store_path):
            print('Track file does not already exist. CREATING...')
            run_interpolation=True

    ssha, ssha_var, ssha_SMOOTHED, ssha_var_SMOOTHED, run_time = run_gpsat(track_df, train_leads_df,
                                                                       experts_df, store_path,
                                                                       run_interp=run_interpolation)
    track_df['f_gpsat'] = ssha
    track_df['f_var_gpsat'] = ssha_var
    track_df['f_gpsat_SMOOTHED'] = ssha_SMOOTHED
    track_df['f_var_gpsat_SMOOTHED'] = ssha_var_SMOOTHED

    return track_df, run_time

def select_experts(leads_df, spacing=25e3, max_experts=128):
    '''
    Selects local experts for GPSat interpolation
    '''
    try:
        dist_along_track = leads_df['dist_along_track'].copy()
    except KeyError as ke:
        print(f'{ke}: Make sure distance along track is def')
        return

    #make sure values are valid
    valid = np.isfinite(dist_along_track)

    dist_along_track = dist_along_track[valid]

    if len(dist_along_track) == 0:
        return pd.DataFrame()
        
    
    grid = np.arange(0, dist_along_track.max(), spacing)
    selected = []
    used = set()

    for target in grid:
        if len(selected) >= max_experts:
            print('Expert limit reached.')
            break
        #subtract each target dist from the dist of each lead, and so find
        #the closest lead to target from the smallest value
        idx = (dist_along_track-target).abs().idxmin()
        #select lead and add to selected and used
        if idx not in used:
            selected.append(idx)
            used.add(idx)
    
    if len(selected) < 3:
        #if we have too few experts
        #find the unselected leads
        remaining_idx = leads_df.index.difference(selected)
        #find no. experts to add
        n_to_add = max_experts-len(remaining_idx)
        #add indices evenly spaced from remaining unselected indices
        step = max(1, len(remaining_idx) // n_to_add)
        idx_added = remaining_idx[::step][:n_to_add]

        selected.extend(list(idx_added))

    experts_df = leads_df.loc[selected]

    return experts_df

def run_gpsat(track_df, leads_df, experts_df, store_path, run_interp = True):
    '''
    Configures GPSat run parameters and executes.
    '''

    if run_interp==True:
        # Notebook-style config: 3D coords, 400km, 100-200km lengthscales
        coords_col = ["x", "y", "t"]
        local_select = [
            {"col": ["x", "y"], "comp": "<=", "val": 400_000},
            {"col": "t", "comp": "<=", "val": 1.0},
            {"col": "t", "comp": ">=", "val": -1.0}
        ]
        data_config = {
            "data_source": leads_df,
            "obs_col": "z",
            "coords_col": coords_col,
            "local_select": local_select,
            "global_select": [{"col": "lat", "comp": ">=", "val": 45}]
        }
        expert_config = {"source" : experts_df}
        pred_config = {"method": "from_dataframe", "df": track_df,
                    "max_dist" : 400_000}
        # Notebook-style: 100-200km lengthscales for 3D
        length_low = [100_000, 100_000, 1e-08]
        length_high = [200_000, 200_000, 4]
        coords_scale = [10_000, 10_000, 1]

        model_config = {
        "oi_model": "GPflowGPRModel",
        "init_params": {"coords_scale": coords_scale, "minibatch_size": 100},
        "constraints": {
            "lengthscales": {"low": length_low, "high": length_high},
            "likelihood_variance": {"low": 0.001, "high": 10}  # Lower noise floor for better endpoint adherence
            }
        }
        # Full mode: enable smoothing (3D coords have x/y)
        locexp = LocalExpertOI(expert_config, data_config, model_config, pred_config)
        locexp.run(store_path=store_path, optimise=True)
        dfs, _ = get_results_from_h5file(store_path)
        smooth_hyperparameters(
            result_file=store_path,
            params_to_smooth=["lengthscales", "kernel_variance", "likelihood_variance"],
            smooth_config_dict={
                "lengthscales": dict(l_x=200_000, l_y=200_000, max=12),
                "likelihood_variance": dict(l_x=200_000, l_y=200_000),
                "kernel_variance": dict(l_x=200_000, l_y=200_000, max=0.1)
            },
            save_config_file=False
        )
        model_config['load_params'] = {"file": store_path, "table_suffix": "_SMOOTHED"}
        locexp = LocalExpertOI(expert_config, data_config, model_config, pred_config)
        locexp.run(store_path=store_path, optimise=False, predict=True, table_suffix='_SMOOTHED')

    # Extract results
    dfs, _ = get_results_from_h5file(store_path)
    preds = dfs["preds"]

    # Glue
    preds['xprt_locs'] = np.sqrt((preds['x'] - preds['x'].iloc[0])**2 + (preds['y'] - preds['y'].iloc[0])**2)
    preds['pred_locs'] = np.sqrt((preds['pred_loc_x'] - preds['pred_loc_x'].iloc[0])**2 + 
                                    (preds['pred_loc_y'] - preds['pred_loc_y'].iloc[0])**2)
    glue_max = 400_000  # Full mode uses 400km like notebook

    glued = glue_local_predictions_1d(preds, "pred_locs", "xprt_locs", ["f*", "f*_var"], glue_max)
    ssha = glued['f*'].values
    ssha_var = glued['f*_var'].values if 'f*_var' in glued.columns else None

    preds_SMOOTHED = dfs["preds_SMOOTHED"]
    run_details = dfs['run_details']
    run_details_SMOOTHED = dfs['run_details_SMOOTHED']

    #measure interpolation time
    run_time = run_details['run_time']
    run_time_SMOOTHED = run_details_SMOOTHED['run_time']

    total_time = run_time.cumsum()
    total_time_SMOOTHED = run_time_SMOOTHED.cumsum()

    final_total_time = total_time.iloc[-1] + total_time_SMOOTHED.iloc[-1]

    # Glue
    preds_SMOOTHED['xprt_locs'] = np.sqrt((preds_SMOOTHED['x'] - preds_SMOOTHED['x'].iloc[0])**2 + (preds_SMOOTHED['y'] - preds_SMOOTHED['y'].iloc[0])**2)
    preds_SMOOTHED['pred_locs'] = np.sqrt((preds_SMOOTHED['pred_loc_x'] - preds_SMOOTHED['pred_loc_x'].iloc[0])**2 + 
                                    (preds_SMOOTHED['pred_loc_y'] - preds_SMOOTHED['pred_loc_y'].iloc[0])**2)
    
    glued_SMOOTHED = glue_local_predictions_1d(preds_SMOOTHED, "pred_locs", "xprt_locs", ["f*", "f*_var"], glue_max)
    ssha_SMOOTHED = glued_SMOOTHED['f*'].values
    ssha_var_SMOOTHED = glued_SMOOTHED['f*_var'].values if 'f*_var' in glued_SMOOTHED.columns else None

    # Handle length mismatch
    if len(ssha) < len(track_df):
        full = np.full(len(track_df), np.nan)
        full_var = np.full(len(track_df), np.nan)
        full[:len(ssha)] = ssha
        full_var[:len(ssha_var)] = ssha_var

        full_SMOOTHED = np.full(len(track_df), np.nan)
        full_var_SMOOTHED = np.full(len(track_df), np.nan)
        full_SMOOTHED[:len(ssha_SMOOTHED)] = ssha_SMOOTHED
        full_var_SMOOTHED[:len(ssha_var_SMOOTHED)] = ssha_var_SMOOTHED

        return full, full_var, full_SMOOTHED, full_var_SMOOTHED, final_total_time
    
    return ssha[:len(track_df)], ssha_var[:len(track_df)], ssha_SMOOTHED[:len(track_df)], ssha_var_SMOOTHED[:len(track_df)], final_total_time

In [5]:
tracks = data_df['track'].unique()
print(len(tracks))
tracks = tracks[:300]

# select for first 300 tracks
data_df = data_df.loc[data_df['track'].isin(tracks)]

4638


In [6]:
plot_data_df, results_df, metrics_df = random_holdout_test(data_df, gpsat_experts=128, replace_previous=False)

Interpolating track 1243, 1 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1244, 2 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1245, 3 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on exper

/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1248, 6 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1249, 7 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert

/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Interpolating track 1272, 27 of 300...
Too few leads for linear interpolation, returning NaN
Too few experts for GPSat, returning NaN
GPSat output invalid or incomplete, skipping...
Interpolating track 1273, 28 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1274, 29 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaN

...

Interpolating track 1521, 267 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...


/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Interpolating track 1522, 268 of 300...
Too few leads for linear interpolation, returning NaN
Too few experts for GPSat, returning NaN
GPSat output invalid or incomplete, skipping...
Interpolating track 1523, 269 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1524, 270 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test 

/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mhen/miniconda3/envs/geol0069_gpsat/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Interpolating track 1548, 294 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1549, 295 of 300...
Track file already exists, KEEPING...
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
Test NaNs — Linear: 0, GPSat: 0
Track interpolated!
Saving...
Interpolating track 1550, 296 of 300...
Insufficient leads (2) for random holdout, skipping...
Interpolating track 1551, 297 o